# OpenPlaque RCA Source-CCTA Seeded Tracer

Clean implementation built from `main` only. Use **Runtime → Run all** once. Google Drive mounts first. The notebook loads source CCTA series **7** and then leaves an interactive viewer on screen.

In the viewer: move to an RCA slice, click the lumen center, press **Use click as OSTIUM**; move distally, click the RCA again, press **Use click as DISTAL**; then press **Trace RCA**. No coordinate typing or additional cell execution is required.

Research prototype only. Every traced path must be visually checked.

In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
# Clone the clean branch and install only dependencies needed here.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib ipympl ipywidgets

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib widget

SRC = Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from openplaque.study import OpenPlaqueStudy
from openplaque.source_centerline import trace_seeded_coronary
print('Dependencies ready.')

## Load source CCTA series 7
The study ZIP is copied from Drive to local Colab storage before extraction/scanning.

In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_seeded_rca'
SOURCE_SERIES = 7
if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f'Missing {DRIVE_ZIP}')

t = time.time()
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying Full_DICOM.zip ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB) to local disk...', flush=True)
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
    print(f'Copy finished in {time.time()-t:.1f}s', flush=True)
else:
    print('Local ZIP already staged.')

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
print('Extracting/scanning DICOM locally...', flush=True)
t = time.time()
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
print(f'Scan finished in {time.time()-t:.1f}s; {len(study.series)} series found.', flush=True)
source_img, source, source_files = study.load_series(SOURCE_SERIES)
print('Source series:', SOURCE_SERIES)
print('Shape zyx:', source.shape)
print('Spacing xyz mm:', source_img.GetSpacing())


## Interactive seed picker
The initial slice is near the aortic root for this UCLA case. Use the slider freely. Click directly on the bright RCA lumen. The second seed should be clearly distal enough that the traced path extends beyond 50 mm.

In [ ]:
state = {'last_click': None, 'ostium': None, 'distal': None, 'result': None}
z_slider = widgets.IntSlider(value=min(335, source.shape[0]-1), min=0, max=source.shape[0]-1, step=1, description='z slice', continuous_update=False, layout=widgets.Layout(width='700px'))
ostium_button = widgets.Button(description='Use click as OSTIUM', button_style='success')
distal_button = widgets.Button(description='Use click as DISTAL', button_style='info')
trace_button = widgets.Button(description='Trace RCA', button_style='warning')
status = widgets.HTML(value='<b>Click the RCA lumen in the image.</b>')
out = widgets.Output()

fig, ax = plt.subplots(figsize=(7,7))
im = ax.imshow(source[z_slider.value], cmap='gray', vmin=-200, vmax=800)
cross = ax.scatter([], [], s=70, marker='+')
saved_o = ax.scatter([], [], s=70, marker='o', facecolors='none', label='ostium')
saved_d = ax.scatter([], [], s=70, marker='s', facecolors='none', label='distal')
ax.set_title(f'Source CCTA series 7 — z={z_slider.value}')
ax.axis('off')
ax.legend(loc='lower right')

def refresh(z):
    im.set_data(source[z])
    ax.set_title(f'Source CCTA series 7 — z={z}')
    # Show saved seeds only when their slice is the current slice.
    if state['ostium'] is not None and state['ostium'][0] == z:
        saved_o.set_offsets([[state['ostium'][2], state['ostium'][1]]])
    else:
        saved_o.set_offsets(np.empty((0,2)))
    if state['distal'] is not None and state['distal'][0] == z:
        saved_d.set_offsets([[state['distal'][2], state['distal'][1]]])
    else:
        saved_d.set_offsets(np.empty((0,2)))
    fig.canvas.draw_idle()

def on_z(change):
    refresh(int(change['new']))
z_slider.observe(on_z, names='value')

def on_click(event):
    if event.inaxes != ax or event.xdata is None or event.ydata is None:
        return
    z = int(z_slider.value); y = int(round(event.ydata)); x = int(round(event.xdata))
    y = int(np.clip(y,0,source.shape[1]-1)); x = int(np.clip(x,0,source.shape[2]-1))
    state['last_click'] = (z,y,x)
    cross.set_offsets([[x,y]])
    status.value = f'Last click: <b>zyx={state["last_click"]}</b>, HU={float(source[z,y,x]):.0f}. Choose OSTIUM or DISTAL.'
    fig.canvas.draw_idle()
fig.canvas.mpl_connect('button_press_event', on_click)

def save_seed(which):
    if state['last_click'] is None:
        status.value = '<b>Click a point first.</b>'
        return
    state[which] = tuple(state['last_click'])
    status.value = f'Saved {which}: <b>{state[which]}</b>. Ostium={state["ostium"]}; distal={state["distal"]}'
    refresh(z_slider.value)
ostium_button.on_click(lambda b: save_seed('ostium'))
distal_button.on_click(lambda b: save_seed('distal'))

def physical_to_zyx(pt_xyz):
    x,y,z = source_img.TransformPhysicalPointToContinuousIndex(tuple(float(v) for v in pt_xyz))
    return np.array([z,y,x], dtype=float)

def show_trace(result):
    route = np.asarray(result.points_zyx_voxel)
    margin = np.ceil(12.0 / np.asarray(source_img.GetSpacing())[::-1]).astype(int)
    lo = np.maximum(0, np.floor(route.min(axis=0)).astype(int)-margin)
    hi = np.minimum(np.asarray(source.shape), np.ceil(route.max(axis=0)).astype(int)+margin+1)
    crop = source[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    rr = route - lo
    fig2, axes = plt.subplots(1,3,figsize=(16,5))
    axes[0].imshow(np.max(crop,axis=0),cmap='gray',vmin=-100,vmax=700); axes[0].plot(rr[:,2],rr[:,1],'-',lw=2); axes[0].set_title('Axial projection')
    axes[1].imshow(np.max(crop,axis=1),cmap='gray',vmin=-100,vmax=700,aspect='auto'); axes[1].plot(rr[:,2],rr[:,0],'-',lw=2); axes[1].set_title('Coronal projection')
    axes[2].imshow(np.max(crop,axis=2),cmap='gray',vmin=-100,vmax=700,aspect='auto'); axes[2].plot(rr[:,1],rr[:,0],'-',lw=2); axes[2].set_title('Sagittal projection')
    for d,pt in sorted(result.landmarks_xyz_mm.items()):
        q = physical_to_zyx(pt)-lo
        axes[0].scatter([q[2]],[q[1]],s=70); axes[0].text(q[2]+2,q[1],f'{d:.0f} mm')
        axes[1].scatter([q[2]],[q[0]],s=70); axes[1].text(q[2]+2,q[0],f'{d:.0f} mm')
        axes[2].scatter([q[1]],[q[0]],s=70); axes[2].text(q[1]+2,q[0],f'{d:.0f} mm')
    for a in axes: a.axis('off')
    plt.tight_layout(); plt.show()

def do_trace(button):
    if state['ostium'] is None or state['distal'] is None:
        status.value = '<b>Set both OSTIUM and DISTAL seeds first.</b>'
        return
    with out:
        clear_output(wait=True)
        print('Tracing RCA. Frangi vesselness + shortest path may take 1–3 minutes...')
        t=time.time()
        try:
            result = trace_seeded_coronary(source, source_img, state['ostium'], state['distal'])
            state['result']=result
            print(f'Done in {time.time()-t:.1f}s. Centerline length={result.length_mm:.1f} mm')
            print('Landmarks available:', sorted(result.landmarks_xyz_mm))
            if 50.0 not in result.landmarks_xyz_mm:
                print('FAIL: traced path is shorter than 50 mm. Pick a more distal RCA seed.')
            show_trace(result)
            print('PASS only if the line follows the RCA continuously and 10/50-mm points are anatomically plausible.')
        except Exception as e:
            print('TRACE FAILED:',repr(e))
trace_button.on_click(do_trace)

display(z_slider, widgets.HBox([ostium_button,distal_button,trace_button]), status, fig.canvas, out)
print('Run All is complete. Use the viewer above; no more cells need to be run.')